<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/07-demand-paging-and-memory-management.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Demand Paging and Memory Management**

The previous chapter explained how a legal virtual address is translated and how the processor raises an exception when no usable translation exists. This chapter begins at the **repairable-fault** branch. The address already belongs to a valid region and the access is allowed in principle; the missing question is how the operating system will supply physical storage, preserve sharing semantics, and keep enough memory available for future requests.

Demand paging is therefore not one algorithm. It is a coordinated policy system involving region metadata, backing objects, the page cache, anonymous memory, swap, replacement, physical-page allocators, writeback, and resource accounting. A useful mental model is:

> **Address translation decides what an address means. Memory management decides when its contents exist in RAM, where they come from, and which resident contents may leave.**

The chapter follows that decision from a page fault to a restarted instruction, then widens the view to replacement, thrashing, allocation, huge pages, NUMA placement, and Linux observability. Device-controller mechanics and interrupt-driven I/O remain in the next chapter; here, storage I/O appears only as one possible wait inside memory recovery.

### **From Address Translation to Demand Paging**

**Demand paging** materializes a page when an access first requires it instead of eagerly placing every logically available page in RAM. It is similar to a library that reserves shelf locations for an entire collection but retrieves a volume from storage only when a reader asks for it. The reservation makes the address meaningful; the later retrieval supplies its current contents.

This distinction separates several measurements that are often incorrectly treated as synonyms:

| Quantity | What it describes | Example |
|---|---|---|
| Virtual size | Address ranges that exist in the process's mapping namespace | A 4 GiB anonymous mapping can exist before most pages are touched |
| Committed memory | Potential future memory use accepted by the system's accounting policy | Private writable mappings may contribute even while nonresident |
| Resident set (RSS) | Pages from those mappings currently present in RAM | Only touched code, data, heap, stack, and mapped-file pages |
| Proportional set (PSS) | Resident pages with shared pages divided among their mappers | A shared library page mapped by four processes contributes roughly one quarter to each PSS |
| Swap usage | Anonymous contents retained outside RAM in a swap backing store | A cold private heap page may be nonresident but recoverable |
| Page cache | RAM holding file contents for reads, writes, and mappings | The same file page can serve `read()` and `mmap()` users |

A new mapping may increase virtual size without immediately increasing RSS. A file-backed page fault may increase RSS without reading storage if the requested file offset is already in the page cache. A process can lose an anonymous page from RSS while its data survives in swap. These transitions explain why “the process allocated 1 GiB” is incomplete unless **allocated** is tied to a specific metric.

Demand paging buys three things:

- **lower startup cost** because unused executable and library pages never need to be read;
- **better capacity utilization** because untouched reservations do not need private frames;
- **dynamic sharing and caching** because a resident file page can satisfy multiple processes.

It also moves work onto the first access. That access may now trap, allocate memory, contend on metadata, wait for reclaim, wait for storage, and then restart. Good systems use demand paging where saved work and capacity outweigh first-touch latency; latency-sensitive software may deliberately prefault selected ranges.

![A demand-paging fault is classified, backed, populated, installed, and restarted.](assets/demand-paging-fault-lifecycle.svg){fig-alt="A legal page fault is validated, assigned a zero, copy-on-write, page-cache, or swap source, supplied with a frame, installed atomically, and restarted; invalid accesses leave through a signal path." width="98%"}

*Figure: original explanatory diagram based on the [Linux page-table overview](https://docs.kernel.org/mm/page_tables.html), [Linux memory-management concepts](https://docs.kernel.org/admin-guide/mm/concepts.html), and the [OSTEP demand-paging mechanism chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-beyondphys.pdf).*

The figure is intentionally broader than “read a page from disk.” A fault can be resolved by mapping a shared zero page, allocating and clearing RAM, copying an existing frame, attaching an already-cached file page, or reading nonresident data. Only some paths perform storage I/O.

### **The Page-Fault Lifecycle**

A page fault is a **precise architectural exception**: the processor reports the faulting virtual address and enough cause information for the kernel to reason about the failed access. The exact registers and cause bits differ by architecture, but the recovery questions are portable:

1. Does this address still belong to a valid region?
2. Does that region allow the attempted read, write, or instruction fetch?
3. What logical object and offset should supply the bytes?
4. Is the content already resident somewhere?
5. If a new frame is needed, can one be allocated or reclaimed?
6. Can the PTE be installed without losing a race with another thread?
7. What translation state must be refreshed before execution resumes?

The word **still** matters. Between the processor taking the exception and the handler inspecting process metadata, another thread may call `munmap()`, change permissions, truncate a mapped file, or fault the same page. Kernel implementations therefore use mapping locks, page or folio locks, reference counts, and retry loops. A classroom flowchart looks linear; a production handler repeatedly revalidates assumptions around operations that may block.

#### **Classifying Faults**

The first classification separates bugs and denied accesses from valid lazy work. A useful backing-oriented taxonomy is:

| Fault class | Region state | Missing or protected state | Normal result |
|---|---|---|---|
| Invalid address | No suitable region, bad canonical form, or access beyond an object | No legal mapping exists | Signal such as `SIGSEGV` or `SIGBUS`, or a controlled kernel error |
| Protection violation | Region/PTE denies the requested mode | Write to immutable data, execute-disabled page, supervisor page | Reject unless protection was intentionally used as a software mechanism |
| Anonymous first touch | Valid anonymous region | No private contents exist yet | Map zero page or allocate a cleared frame |
| Copy-on-write | Valid private writable region | Current PTE is deliberately read-only and shared | Allocate/copy or upgrade an exclusive page, then make this mapping writable |
| File-backed miss | Valid mapped-file region | File offset is not mapped in this process | Attach page-cache folio; initiate file read only on cache miss |
| Swapped anonymous page | Valid anonymous region | PTE encodes or leads to a swap identity | Recover contents from swap cache or swap storage |
| Translation bookkeeping | Valid resident page | Accessed/dirty tracking or stale translation requires attention | Update metadata and resume without reconstructing contents |

Linux and many teaching systems call a fault **minor** when it can be completed without reading the requested page from a storage device, and **major** when storage I/O is required. Minor does not mean free: allocating, zeroing, copying, locking, invalidating translations, and scheduling can still be costly. Conversely, a file-backed fault can be minor because another process or an earlier `read()` already populated the page cache.

Two additional distinctions prevent misleading diagnoses:

- a **TLB miss** usually triggers a hardware page-table walk and is not itself a page fault;
- a **page fault** is not automatically evidence of insufficient RAM, because first-touch and COW faults are expected even on an idle machine.

#### **Locating, Loading, and Installing a Page**

Once the access is legal, the kernel derives a **page identity** from the region. For anonymous first touch, the identity may be “new zero-filled page at this virtual offset.” For a file mapping it is approximately `(file object, page-aligned file offset)`. For a swapped page it includes a swap entry. For COW it includes the currently shared source page plus the requirement that the writer receive private semantics.

The recovery path then separates **finding contents** from **finding a frame**:

1. Look for the logical page in the appropriate cache or metadata structure.
2. If a suitable resident page already exists, acquire a stable reference to it.
3. Otherwise request a physical frame under the required node, zone, size, and allocation constraints.
4. If the free lists cannot satisfy the request, invoke reclaim and possibly compaction.
5. Populate the frame by clearing, copying, or reading its backing source.
6. Wait for in-progress population if another thread owns the same page operation.
7. Recheck that the region and desired PTE state are still valid.
8. Install the PTE with the intended frame number, permissions, and software state.
9. Release locks and references in an order that prevents either premature reuse or stale mappings.

For file pages, the page cache ensures that concurrent faults on the same file offset normally converge on one cached page rather than issue unrelated reads and create incoherent copies. One thread may create and lock the cache entry, another may find it and wait, and both can later map the same completed contents.

A replacement decision is also content-aware. A clean file page can often be discarded because the file remains its authoritative backing. A dirty file page needs writeback. A dirty anonymous page needs swap or must stay resident. A pinned page may not be reclaimable at all. “Choose any old page” is therefore not a complete allocator slow path.

The simplified kernel-style pseudocode below emphasizes retries and ownership:

```text
handle_fault(task, address, access):
    repeat:
        region = lookup_and_lock_region(task.address_space, address)
        if region is absent or access violates region.policy:
            return deliver_memory_signal(task, address, access)

        identity, action = classify_backing(region, address, access)
        page = find_or_begin_population(identity)

        if page says "another thread is populating":
            release_region_lock()
            wait_for_page(page)
            continue

        if action needs a new frame:
            frame = allocate_frame_with_reclaim(region.constraints)
            if frame is unavailable:
                release_region_lock()
                return allocation_failure_or_oom()
            populate(frame, action, identity)  // zero, copy, file, or swap

        if region_changed_while_work_could_block():
            discard_temporary_result_safely()
            release_region_lock()
            continue

        install_pte_atomically(address, page_or_frame, permissions(action))
        release_region_lock()
        return restart_faulting_instruction
```

Real Linux code is considerably more specialized, but this model captures the proof obligations: authorize against current metadata, establish one coherent page, publish the mapping atomically, and clean up every losing or failing path.

#### **Restarting the Faulting Instruction**

The handler usually does not emulate the user instruction. It repairs the execution environment and returns through the architecture's exception-return path. The processor restores the saved program counter and privilege state, retries the original instruction, and translation now succeeds.

This works because the exception is **precise**: architectural state identifies the instruction that did not complete as if it had completed normally. A complex instruction may perform enough internal work that the architecture and kernel must preserve restart semantics carefully, but user space observes either the defined completed operation or an exception/signal, not a casually half-published memory access.

Restart also explains why handlers must make progress. If the PTE is installed with the wrong permission, if a stale translation remains usable, or if the region changed, the same instruction can fault again. Repeated faults are not automatically a kernel bug: a signal handler, concurrent mapping change, user-space pager, or changing resource condition can legitimately intervene. The invariant is that each retry revalidates the current address-space contract.

The latency of a fault can dominate ordinary memory access even when faults are rare. If a normal reference costs $t_m$, a fault occurs with probability $p$, and average fault service costs $t_f$, a deliberately simplified expectation is:

$$
E[T] = (1-p)t_m + p(t_f + t_m)
$$

Here $t_f$ includes exception entry, kernel work, possible scheduling and I/O, while the final $t_m$ is the retried access. With $t_m = 100$ ns and $t_f = 1$ ms, even $p=10^{-5}$ adds roughly $10$ ns per reference on average. The formula is not a queueing model, but it shows why major-fault probability and tail latency matter disproportionately.

### **Lazy Allocation and Zero-Fill-on-Demand**

An anonymous mapping promises bytes that initially read as zero. Eagerly allocating and clearing every page at `mmap()` or heap-growth time would waste RAM and CPU for pages that may never be touched. **Lazy allocation** records the region but delays private frame creation until access makes it necessary.

A common optimization separates the first read from the first write:

- an untouched anonymous page can read through a shared, immutable **zero page**;
- a write cannot modify that global frame, so it raises a write-protection fault;
- the handler allocates a private frame, ensures it is zeroed, installs a writable PTE, and restarts the store.

![Lazy anonymous memory can share a zero page until the first write.](assets/lazy-zero-first-touch-animated.svg){fig-alt="A reserved anonymous virtual range has no private frames, its first read can map an immutable shared zero page, and its first write faults into a private cleared writable frame." width="96%"}

*Figure: original explanatory diagram based on the Linux [no-MMU/MMU mapping discussion](https://docs.kernel.org/admin-guide/mm/nommu-mmap.html), [`madvise()` semantics](https://man7.org/linux/man-pages/man2/madvise.2.html), and [`/proc/kpageflags` zero-page description](https://docs.kernel.org/admin-guide/mm/pagemap.html).*

Zeroing is also a security boundary. A physical frame previously used by another process may contain secrets. Before user space can observe that frame, the kernel must clear it or otherwise prove the requested backing object completely overwrites it. Laziness changes **when** clearing happens, not whether isolation is required.

The following Linux program reserves anonymous memory, uses `mincore()` to take residency snapshots, touches selected pages, and then gives pages back with `MADV_DONTNEED`. It is an observation tool, not a portable claim about exact counts: transparent huge pages, pre-faulting, the shared zero page, and concurrent reclaim can change what a particular kernel reports.

<details>
<summary><strong>C: observe first touch with mmap, mincore, and madvise</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/mman.h>
#include <unistd.h>

static size_t resident_pages(void *base, size_t length, size_t page_size) {
    size_t pages = (length + page_size - 1) / page_size;
    unsigned char *state = calloc(pages, 1);
    if (state == NULL) {
        perror("calloc");
        exit(EXIT_FAILURE);
    }

    // mincore() is a point-in-time residency query, not an ownership query.
    if (mincore(base, length, state) == -1) {
        perror("mincore");
        free(state);
        exit(EXIT_FAILURE);
    }

    size_t resident = 0;
    for (size_t i = 0; i < pages; ++i) {
        resident += (state[i] & 1U) != 0;
    }
    free(state);
    return resident;
}

int main(void) {
    long result = sysconf(_SC_PAGESIZE);
    if (result <= 0) {
        fputs("cannot determine page size\n", stderr);
        return EXIT_FAILURE;
    }

    size_t page_size = (size_t)result;
    size_t pages = 64;
    size_t length = pages * page_size;
    unsigned char *region = mmap(NULL, length, PROT_READ | PROT_WRITE,
                                 MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (region == MAP_FAILED) {
        perror("mmap");
        return EXIT_FAILURE;
    }

    printf("after mmap:       %zu / %zu pages reported resident\n",
           resident_pages(region, length, page_size), pages);

    // Read one byte from every fourth page. Reads may use a shared zero page.
    volatile unsigned int sum = 0;
    for (size_t i = 0; i < pages; i += 4) {
        sum += region[i * page_size];
    }
    printf("after sparse read: %zu / %zu pages reported resident\n",
           resident_pages(region, length, page_size), pages);

    // Writing every fourth page requires writable private contents.
    for (size_t i = 0; i < pages; i += 4) {
        region[i * page_size] = (unsigned char)i;
    }
    printf("after sparse write:%zu / %zu pages reported resident\n",
           resident_pages(region, length, page_size), pages);

    // For private anonymous memory, DONTNEED discards current contents;
    // later accesses once again observe zero-filled pages.
    if (madvise(region, length, MADV_DONTNEED) == -1) {
        perror("madvise");
    }
    printf("after DONTNEED:    %zu / %zu pages reported resident\n",
           resident_pages(region, length, page_size), pages);

    if (munmap(region, length) == -1) {
        perror("munmap");
        return EXIT_FAILURE;
    }
    return sum != 0;  // keeps the reads observable to the compiler
}
```

```bash
cc -std=c11 -O2 -Wall -Wextra -Wpedantic first_touch.c -o first_touch
./first_touch
```

</details>

`mincore()` reports whether pages are resident at the instant of the call; it does not reveal whether the process owns a private frame, shares the zero page, or will keep that page resident. The [Linux `mincore(2)` manual](https://man7.org/linux/man-pages/man2/mincore.2.html) and [`madvise(2)` manual](https://man7.org/linux/man-pages/man2/madvise.2.html) define those APIs. For footprint attribution, `/proc/PID/smaps` or `smaps_rollup` is more informative.

Lazy allocation is excellent when a program reserves more than it touches. It is less attractive when the first-touch pause falls on a latency-critical path, or when a late allocation failure is harder to handle than an early reservation failure. `MADV_POPULATE_READ`, `MADV_POPULATE_WRITE`, `MAP_POPULATE`, memory locking, and application-level warm-up represent different ways to move or constrain that cost; none should be enabled blindly.

### **Copy-on-Write**

**Copy-on-write (COW)** preserves the appearance of private writable memory while physically sharing pages until a write would make the sharers disagree. The canonical case is `fork()`: the child begins with the parent's memory contents, but duplicating every anonymous page before either process runs would be expensive and often pointless because the child may immediately call `exec()`.

The kernel can instead:

1. let parent and child PTEs point to the same physical page;
2. remove write permission from both mappings and mark the relationship as COW in software-visible state;
3. maintain references so the frame cannot be reused while either mapping depends on it;
4. on a write fault, determine whether the page remains shared;
5. allocate a new frame and copy contents when separation is required;
6. point only the writer's PTE at the new frame, make it writable, invalidate stale translation state, and restart the store.

![A child write converts one shared COW page into two private pages.](assets/cow-private-write-animated.svg){fig-alt="Parent and child initially point through read-only COW PTEs to one physical frame; a child write faults, copies the data to a new frame, remaps only the child writable, and leaves the parent value unchanged." width="98%"}

*Figure: original explanatory diagram based on the [`fork()` specification](https://pubs.opengroup.org/onlinepubs/9799919799/functions/fork.html), the Linux [`madvise()` fork controls](https://man7.org/linux/man-pages/man2/madvise.2.html), and [Linux page-table documentation](https://docs.kernel.org/mm/page_tables.html). It is newly drawn for this chapter rather than reusing the process-lifecycle overview from chapter 02.*

If the writer is now the only owner, a sophisticated handler may avoid copying and simply upgrade that mapping to writable. Conversely, COW can amplify cost after a large `fork()`: writing one byte in each shared page can allocate and copy the entire private working set. Memory limits and accounting must therefore reason about potential COW breakage, not just current sharing.

This small program demonstrates the semantic guarantee. Parent and child begin from one anonymous value, the child writes `99`, and the parent still reads `41`. The output alone cannot prove that the kernel used COW internally, but it shows the behavior COW efficiently implements.

<details>
<summary><strong>C: observe private memory semantics across fork</strong></summary>

```c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <sys/mman.h>
#include <sys/wait.h>
#include <unistd.h>

int main(void) {
    long result = sysconf(_SC_PAGESIZE);
    if (result <= 0) {
        fputs("cannot determine page size\n", stderr);
        return EXIT_FAILURE;
    }

    size_t page_size = (size_t)result;
    int *value = mmap(NULL, page_size, PROT_READ | PROT_WRITE,
                      MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (value == MAP_FAILED) {
        perror("mmap");
        return EXIT_FAILURE;
    }
    *value = 41;  // Materialize the page before fork().

    int ready[2];
    if (pipe(ready) == -1) {
        perror("pipe");
        munmap(value, page_size);
        return EXIT_FAILURE;
    }

    pid_t pid = fork();
    if (pid == -1) {
        perror("fork");
        return EXIT_FAILURE;
    }

    if (pid == 0) {
        close(ready[0]);
        *value = 99;  // Normally triggers COW for the child's mapping.
        dprintf(STDOUT_FILENO, "child sees  %d\n", *value);
        if (write(ready[1], "x", 1) != 1) {
            _exit(2);
        }
        close(ready[1]);
        _exit(0);
    }

    close(ready[1]);
    char byte;
    if (read(ready[0], &byte, 1) != 1) {
        fputs("child did not signal completion\n", stderr);
    }
    close(ready[0]);

    // The child's store changed its private mapping, not this one.
    printf("parent sees %d\n", *value);
    if (waitpid(pid, NULL, 0) == -1) {
        perror("waitpid");
    }

    if (munmap(value, page_size) == -1) {
        perror("munmap");
        return EXIT_FAILURE;
    }
    return EXIT_SUCCESS;
}
```

```bash
cc -std=c11 -O2 -Wall -Wextra -Wpedantic cow.c -o cow
./cow
```

</details>

The expected values are `child sees 99` and `parent sees 41`; line order is coordinated by the pipe for readability. `fork()` remains a concurrent operation, and production code must use async-signal-safe operations between `fork()` and `exec()` in a multithreaded child. That process-control rule is separate from COW's memory semantics.

### **Memory-Mapped Files and Shared Pages**

`mmap()` makes a file object's offsets directly addressable through virtual memory. The mapping is lazy: establishing it does not prove that all bytes are resident. On first access, the page-fault handler translates the virtual offset into a file offset and consults the **page cache**, which is the normal meeting point for buffered file reads, writes, and mappings in Linux.

![Buffered file I/O and memory mappings converge on the page cache.](assets/page-cache-mmap-sharing.svg){fig-alt="read and write system calls, shared mappings, and private mappings all consult cached file pages; shared writes dirty the cached page while private writes create an anonymous COW page." width="98%"}

*Figure: original explanatory diagram based on the [Linux page-cache documentation](https://docs.kernel.org/mm/page_cache.html) and POSIX [`mmap()` mapping-type semantics](https://pubs.opengroup.org/onlinepubs/9799919799/functions/mmap.html).*

The mapping type determines the disposition of writes:

| Mapping | Read path | Write path | Effect on file |
|---|---|---|---|
| `MAP_SHARED` | Map the cached file page | Modify shared cached state and mark it dirty | Changes are eligible for writeback to the underlying object |
| `MAP_PRIVATE` | Initially map file contents | Break away through COW into private anonymous memory | Private modifications do not update the file object |
| Shared anonymous | Map a shared memory object such as shmem/tmpfs | Processes intentionally observe shared modifications | No ordinary persistent file is required |

“Shared” does not mean unsynchronized. Two threads or processes that write the same mapped bytes still need an application-level protocol such as atomics, process-shared mutexes, or ownership rules. Mapping creates visibility; it does not create a data-structure invariant.

Likewise, **dirty**, **written back**, and **durable** are different states. A store to `MAP_SHARED` can update a resident cached page quickly. The kernel may write it to a filesystem later. Filesystem journaling, device caches, ordering barriers, `msync()`, `fsync()`, and application protocols determine when durable recovery is promised. Those persistence mechanisms belong to the file-system chapter, but the distinction matters here: residency is not durability.

Mapped files also carry failure modes that ordinary pointer intuition hides:

- accessing a page beyond the current valid end of a truncated file can raise `SIGBUS`;
- concurrent file modification and mapping access require a defined synchronization contract;
- a mapping can survive after its file descriptor is closed because the mapping holds its own object reference;
- `munmap()` removes mappings but does not by itself promise durable writeback;
- direct I/O can bypass the ordinary page-cache path and introduces coherence constraints with cached access.

The central advantage is unification. One cached file page can be read through a system call, mapped by several processes, and discarded under memory pressure because the file remains a reconstructible backing source. The central risk is that a pointer access can now trigger blocking work and file-related signals, so latency and error handling cannot be reasoned about as if every load were already backed by anonymous RAM.

### **Page Replacement**

When a fault or kernel allocation needs a frame and no suitable free block is immediately available, the system attempts **reclaim**: convert some currently occupied frames back into allocatable frames. **Page replacement** is the policy problem inside reclaim that asks which eligible resident page should lose its frame.

The ideal victim is cheap to remove and unlikely to be referenced again soon. Those goals pull in several signals:

- **recency and frequency** estimate future reuse from past access;
- **backing type** determines whether contents can be discarded, written to a file, or placed in swap;
- **dirty state** determines whether writeback is required;
- **sharing and reference counts** determine how many mappings benefit from the page;
- **reclaimability** excludes pinned, locked, or otherwise unevictable pages;
- **scope and accounting** determine which workload is allowed to pay for the new allocation;
- **NUMA placement and page size** constrain which frame would actually satisfy the request.

Thus, a replacement algorithm from a textbook normally receives an abstract set of equal-cost pages. A real reclaimer first builds and filters candidate sets, may initiate asynchronous writeback, and can fail to produce the right contiguous order even after freeing many base pages.

At an abstract level, an online replacement policy has this interface:

```text
on_reference(page):
    update the policy's history for page

on_miss(requested_page):
    if a free frame exists:
        install requested_page
    else:
        victim = choose among eligible resident pages
        preserve victim contents if required
        remove victim mapping(s)
        install requested_page
```

The policy should be evaluated on a **reference string** with a fixed frame count and initial state. Comparing fault counts from different traces or capacities says little about algorithm quality.

#### **Optimal, FIFO, and Random Replacement**

The **Optimal** policy, also called OPT or MIN, evicts the page whose next reference lies farthest in the future. If a resident page is never used again, that page is the best victim. OPT minimizes misses for a known reference string, but an online operating system cannot know arbitrary future references. Its value is as a lower bound for simulations and trace-driven studies.

**First-In, First-Out (FIFO)** evicts the page that entered memory earliest. A queue makes selection inexpensive, but residence age is not access recency. A page loaded long ago may be used on every loop iteration and still sit at the queue head. FIFO can also exhibit **Belady's anomaly**: for some reference strings, adding a frame increases rather than decreases the number of faults.

**Random replacement** selects an eligible victim without maintaining recency order. It is simple and avoids some adversarial interactions with deterministic history, but it can discard a hot page by chance. Experiments must fix or report the pseudo-random seed and should compare distributions over seeds rather than one lucky run.

![OPT, FIFO, LRU, and Random define victims from different information.](assets/replacement-policy-trace.svg){fig-alt="Four panels compare optimal, FIFO, exact LRU, and random replacement for the same thirteen-page reference stream and three-frame capacity, including their state requirements and fault-count behavior." width="98%"}

*Figure: original explanatory comparison based on the [OSTEP replacement-policy chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-beyondphys-policy.pdf). Fault counts are for the displayed reference string, three initially empty frames, and the simulator below.*

The following Python code is intentionally a **policy simulator**, not an operating-system implementation. Python is appropriate here because the object being implemented is the mathematical trace model. It records every decision and makes random replacement reproducible.

<details>
<summary><strong>Python: compare OPT, FIFO, LRU, Random, and Clock on one trace</strong></summary>

```python
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from math import inf
from random import Random
from typing import Iterable


@dataclass(frozen=True)
class Step:
    time: int
    page: int
    hit: bool
    victim: int | None
    frames: tuple[int, ...]


def simulate_stack_policy(
    references: Iterable[int],
    capacity: int,
    policy: str,
    *,
    seed: int = 7,
) -> list[Step]:
    """Simulate OPT, FIFO, LRU, or deterministic-seed Random."""
    refs = list(references)
    if capacity <= 0:
        raise ValueError("capacity must be positive")
    if policy not in {"OPT", "FIFO", "LRU", "Random"}:
        raise ValueError(f"unknown policy: {policy}")

    frames: list[int] = []
    fifo_order: deque[int] = deque()
    last_used: dict[int, int] = {}
    rng = Random(seed)
    trace: list[Step] = []

    for time, page in enumerate(refs):
        if page in frames:
            last_used[page] = time
            trace.append(Step(time, page, True, None, tuple(frames)))
            continue

        victim = None
        if len(frames) == capacity:
            if policy == "FIFO":
                victim = fifo_order.popleft()
            elif policy == "LRU":
                victim = min(frames, key=last_used.__getitem__)
            elif policy == "Random":
                victim = rng.choice(frames)
            else:  # OPT: choose the page whose next use is farthest away.
                def next_use(candidate: int) -> float:
                    for future_time in range(time + 1, len(refs)):
                        if refs[future_time] == candidate:
                            return future_time
                    return inf

                victim = max(frames, key=next_use)

            frames.remove(victim)
            last_used.pop(victim, None)

        frames.append(page)
        if policy == "FIFO":
            fifo_order.append(page)
        last_used[page] = time
        trace.append(Step(time, page, False, victim, tuple(frames)))

    return trace


def simulate_clock(references: Iterable[int], capacity: int) -> list[Step]:
    """Simulate second-chance Clock with one reference bit per frame."""
    refs = list(references)
    frames: list[int | None] = [None] * capacity
    referenced = [0] * capacity
    hand = 0
    trace: list[Step] = []

    for time, page in enumerate(refs):
        if page in frames:
            slot = frames.index(page)
            referenced[slot] = 1
            visible = tuple(x for x in frames if x is not None)
            trace.append(Step(time, page, True, None, visible))
            continue

        # Search circularly. Referenced pages receive a second chance.
        while frames[hand] is not None and referenced[hand] == 1:
            referenced[hand] = 0
            hand = (hand + 1) % capacity

        victim = frames[hand]
        frames[hand] = page
        referenced[hand] = 1
        hand = (hand + 1) % capacity
        visible = tuple(x for x in frames if x is not None)
        trace.append(Step(time, page, False, victim, visible))

    return trace


references = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
for name in ["OPT", "FIFO", "LRU", "Random"]:
    run = simulate_stack_policy(references, 3, name, seed=7)
    faults = sum(not step.hit for step in run)
    print(f"{name:6s}: faults={faults}, hits={len(run) - faults}")

clock_run = simulate_clock(references, 3)
clock_faults = sum(not step.hit for step in clock_run)
print(f"{'Clock':6s}: faults={clock_faults}, hits={len(clock_run) - clock_faults}")
```

Representative output for this exact code is:

```text
OPT   : faults=7, hits=6
FIFO  : faults=10, hits=3
LRU   : faults=9, hits=4
Random: faults=11, hits=2
Clock : faults=9, hits=4
```

</details>

One trace cannot rank policies universally. Change the loop structure, phase lengths, capacity, or random seed and the comparison can change. OPT remains a per-trace lower bound; it is not a deployable winner.

#### **LRU and Clock Approximations**

**Least Recently Used (LRU)** evicts the page whose most recent reference is oldest. It relies on temporal locality: pages used recently are more likely to be used again soon. Exact LRU can be implemented for a small software cache with a hash table plus linked list, but applying an ordered-list update to every hardware memory reference would overwhelm the fast path.

Hardware commonly supplies an **accessed** or **referenced** bit. Software can periodically clear or sample it, learning whether a page was touched during an interval rather than the exact order of every touch. **Clock**, or second chance, arranges candidate pages in a conceptual circle:

```text
choose_clock_victim():
    loop:
        page = frame_at(clock_hand)
        if page is not reclaimable:
            advance(clock_hand)
        else if page.referenced == 1:
            page.referenced = 0
            advance(clock_hand)       // grant a second chance
        else:
            victim = page
            advance(clock_hand)
            return victim
```

![Clock clears reference bits while scanning and selects the first eligible zero.](assets/clock-second-chance-animated.svg){fig-alt="An animated clock hand scans four frames around a circle, clears reference bits of recently accessed pages, and selects the first eligible frame whose reference bit is zero." width="94%"}

*Figure: original explanatory animation based on the [OSTEP replacement-policy discussion](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-beyondphys-policy.pdf) and Linux [`/proc/kpageflags` referenced/LRU fields](https://docs.kernel.org/admin-guide/mm/pagemap.html).*

Clock is not exact LRU. A bit says “referenced at least once since it was cleared,” so two pages with very different access counts can look identical. Scan speed determines the aging interval, and a large cold population can make one allocation scan many entries. Implementations therefore batch work, maintain multiple lists or generations, and incorporate page type and reclaim cost.

Modern Linux can use **Multi-Gen LRU**, which groups pages into aging generations instead of maintaining a literal global LRU order. It samples access information, advances generations, and reclaims from older generations while considering anonymous and file-backed behavior. The exact kernel policy evolves; the stable lesson is that practical replacement uses **coarse recency classes** and amortized sampling, not a perfect timestamped order for every load. The current [Linux Multi-Gen LRU documentation](https://docs.kernel.org/admin-guide/mm/multigen_lru.html) describes its controls and pressure behavior.

| Policy | History kept | Per-reference cost | Main teaching value |
|---|---:|---:|---|
| Exact LRU | Total order or timestamp for every page | High at page-table scale | Defines idealized recency behavior |
| Aging counters | Periodic bit samples shifted into counters | Batched periodic work | Distinguishes several recency levels |
| Clock | One/few bits plus circular position | Bit set in hardware; scan on reclaim | Shows second-chance approximation clearly |
| Generational LRU | Age cohorts and sampled references | Batched aging and list movement | Scales recency estimation to modern systems |

#### **Local and Global Replacement**

Replacement also needs a **victim scope**. Under local replacement, a process or resource group evicts only from its assigned resident allocation. Under global replacement, a fault may select any eligible page in a shared pool.

![Local and global reclaim trade isolation against flexible use of capacity.](assets/local-global-reclaim.svg){fig-alt="Local replacement forces workload A to evict from its own quota while protecting workload B; global replacement lets A select a cold page from B, while a memory cgroup forms a bounded shared pool." width="96%"}

*Figure: original explanatory diagram based on the local/global replacement treatment in [University of Wisconsin page-replacement notes](https://pages.cs.wisc.edu/~swift/classes/cs537-fa07/lectures/13-pageReplacement.pdf) and Linux [memory-controller concepts](https://docs.kernel.org/admin-guide/cgroup-v2.html#memory).*

| Scope | Strength | Failure mode |
|---|---|---|
| Strict local | Strong isolation and predictable maximum footprint | A process can thrash while another allocation sits idle |
| System global | Capacity flows toward active workloads | One workload can evict another's hot pages and create cross-service latency |
| Group-local/global-within-group | Flexible sharing inside a controlled budget | Limits and group hierarchy must reflect service ownership |

Linux memory cgroups make the middle position practical: pages are charged to a workload hierarchy, reclaim can occur inside a constrained group, and the whole system still has global pressure behavior. “Which page is cold?” and “whose page may pay?” are separate policy questions.

### **Working Sets, Thrashing, and Memory Pressure**

A process usually references memory in phases: a parser moves through one input, a database changes index levels, and a program enters a different call graph. The **working set** at time $t$ over a recent window of length $\Delta$ is the set of distinct pages referenced in that window:

$$
W_i(t, \Delta) = \{p \mid p \text{ was referenced by process } i \text{ during } (t-\Delta, t]\}
$$

The window is a model parameter, not a hardware constant. If $\Delta$ is too small, it misses useful locality; if too large, it combines obsolete phases. The total active demand can be approximated as:

$$
D(t, \Delta) = \sum_i |W_i(t, \Delta)|
$$

If $D$ persistently exceeds the frames available to those workloads, every policy must leave some actively needed pages nonresident. Pages are evicted, referenced again quickly, reconstructed, and evicted again. This self-reinforcing loss of useful progress is **thrashing**.

![When active working sets exceed available frames, reclaim and faults form a feedback loop.](assets/working-set-thrashing-feedback.svg){fig-alt="A recent reference window defines a working set; when summed active working sets exceed usable frames, active pages are evicted, fault again, stall tasks, reduce useful progress, and feed repeated eviction." width="98%"}

*Figure: original explanatory diagram based on the working-set and thrashing discussion in [OSTEP paging policy](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-beyondphys-policy.pdf) and Linux [PSI documentation](https://docs.kernel.org/accounting/psi.html).*

High memory utilization alone is not thrashing. A healthy system intentionally uses spare RAM as file cache and can reclaim cold pages cheaply. Evidence of thrashing instead combines:

- sustained high fault or refault rate;
- aggressive page scanning and stealing;
- repeated swap-in/swap-out or writeback;
- low application throughput or long latency;
- significant memory stall time, especially system-wide **full** PSI;
- improvement when load is reduced or the memory budget is increased.

**Page-fault frequency (PFF)** is an alternative control idea: measure a workload's fault rate and grant more frames when it is too high, reclaim frames when it is very low, and reduce concurrency if total demand cannot fit. Working-set size estimates demand directly; PFF observes one consequence. Both are feedback policies and both depend on a sensible time scale.

The remedies target different causes. Better replacement helps when cold and hot pages are distinguishable near the capacity boundary. It cannot create capacity when every candidate is hot. In that case, the system must lower active concurrency, increase the memory budget, reduce application working sets, compress or tier cold data, or accept degraded performance.

### **Swap, Overcommit, and Out-of-Memory Handling**

**Swap** provides a recoverable backing location for anonymous memory that leaves RAM. Clean file-backed pages normally do not need swap because their file already reconstructs them. Anonymous contents have no such ordinary file offset, so the kernel must retain their bytes somewhere or refuse to reclaim them.

Swap can improve useful RAM allocation when anonymous data is genuinely cold. It cannot make storage behave like DRAM. If the active working set itself is swapped repeatedly, latency and device traffic rise sharply. Compressed-memory layers such as zswap can trade CPU and compressed RAM for fewer storage transfers, but they remain part of the same capacity and pressure trade-off.

**Overcommit** is a separate promise-accounting question. Processes commonly reserve more writable virtual memory than they simultaneously use. Strictly requiring one RAM-or-swap unit for every reservation wastes capacity, but accepting every promise creates the possibility that future first touches cannot all be honored.

Linux exposes policies through `vm.overcommit_memory` and related controls. In strict accounting mode, the documented commit limit is approximately:

$$
CommitLimit = (RAM - reserved\ hugeTLB)\times\frac{overcommit\ ratio}{100} + swap
$$

`Committed_AS` estimates accepted commitments; it is not RSS and does not predict which process will touch memory next. The [Linux overcommit documentation](https://docs.kernel.org/mm/overcommit-accounting.html) defines which mappings are accounted and the available modes. Configuration is a reliability contract: a scientific batch node, interactive laptop, database host, and latency-critical service may rationally choose different risk levels.

![Allocation failure escalates through reclaim, writeback or swap, compaction, and scoped OOM handling.](assets/memory-pressure-swap-oom.svg){fig-alt="A frame request checks accounting and free blocks, tries reclaim and possible compaction, returns on success, or reaches allocation failure and scoped out-of-memory handling; side panels distinguish commit, RSS, swap, and reclaim costs." width="98%"}

*Figure: original explanatory diagram based on Linux [overcommit accounting](https://docs.kernel.org/mm/overcommit-accounting.html), [OOM handling](https://docs.kernel.org/mm/oom.html), [page flags](https://docs.kernel.org/admin-guide/mm/pagemap.html), and the [OSTEP demand-paging mechanism](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-beyondphys.pdf).*

An allocation under pressure can follow several outcomes: wake background reclaim, perform direct reclaim in the requesting context, initiate writeback, swap anonymous pages, compact movable pages for a contiguous request, fall back to another allowed node/zone, return an allocation failure, throttle, or invoke an OOM policy. The exact path depends on allocation flags, order, cgroup limits, privileges, reclaimability, and kernel configuration.

The **OOM killer** is not an ordinary replacement algorithm. It is a last-resort recovery mechanism when a relevant scope cannot satisfy memory obligations and reclaim cannot restore progress. Linux may handle OOM inside a memory cgroup or at system scope. Victim scoring attempts to free substantial memory while respecting adjustments and protected tasks, but termination is inherently disruptive. Applications should still check allocation and mapping errors where APIs report them; operators should treat OOM events as evidence to investigate capacity, limits, leaks, burst assumptions, and workload priority.

| Mechanism | What it controls | What it cannot guarantee |
|---|---|---|
| Demand paging | when a legal page becomes resident | that future frames will always be available |
| Swap | where cold anonymous contents can survive outside RAM | low latency for an active swapped working set |
| Overcommit accounting | which future-use promises are accepted | exact physical footprint or future access order |
| Reclaim | which resident frames become reusable | success for pinned pages or badly fragmented high-order requests |
| OOM policy | how the system regains progress after failure | graceful application-level recovery or perfect victim choice |

### **Physical-Memory Allocation**

Replacement returns pages to free memory; a **physical-memory allocator** organizes those free pages and satisfies constrained requests. User-space `malloc()` operates inside a process address space and usually obtains larger regions from the kernel. The kernel's page allocator instead deals in physical page frames, NUMA nodes, zones, contiguity, and contexts that may or may not sleep or reclaim.

A conceptual page-allocation ADT is:

```text
allocate_pages(order, constraints) -> aligned block or failure
free_pages(block, order)
```

An order-$k$ block contains $2^k$ contiguous base pages. Constraints can express an allowed NUMA node set, a zone reachable by a device, whether reclaim or filesystem I/O is permitted, whether zeroing is required, and whether the request may fall back. A machine can have many free pages in total yet fail an order-$k$ request because no suitable contiguous aligned block remains.

Linux maintains per-node and per-zone state, free-area lists, watermarks, and fallback orders. Background reclaim tries to protect reserves before allocations reach emergency conditions. These details make “free memory” multidimensional: amount, location, order, mobility, and context all matter.

#### **Buddy Allocation**

The **binary buddy allocator** organizes free physical memory in power-of-two blocks. If no block of the requested order exists, it splits a larger block into two equal **buddies**, repeats until reaching the target order, and returns one half. On free, it checks whether the unique aligned buddy of the same order is also free; if so, the pair coalesces and the test repeats at the next order.

![The buddy allocator splits a large power-of-two block and later merges aligned free buddies.](assets/buddy-split-merge-animated.svg){fig-alt="An eight-page order-three block splits into order-two and order-one blocks to satisfy a two-page request, then the animated free path merges a block with its same-order aligned buddy." width="96%"}

*Figure: original explanatory animation based on the Linux [`/proc/kpageflags` BUDDY description](https://docs.kernel.org/admin-guide/mm/pagemap.html) and the kernel.org [physical-page allocation overview](https://www.kernel.org/doc/gorman/html/understand/understand009.html). The latter documents an older Linux implementation, so the figure uses only the enduring buddy algorithm, not obsolete structure names.*

For a block beginning at page-frame index `start`, the buddy at order $k$ is found conceptually by toggling the $k$th block bit:

$$
buddy(start,k)=start\ \operatorname{XOR}\ 2^k
$$

The alignment guaranteed by power-of-two blocks makes that operation possible. Simplified pseudocode is:

```text
allocate(requested_pages):
    target = ceil(log2(requested_pages))
    order = first order >= target with a nonempty free list
    if no such order exists:
        fail
    block = remove_one(free_list[order])
    while order > target:
        order = order - 1
        left, right = split(block)
        insert(right, free_list[order])
        block = left
    return block, target

free(block, order):
    while order < MAX_ORDER:
        buddy = block.start XOR (1 << order)
        if buddy is not free at exactly this order:
            break
        remove(buddy, free_list[order])
        block = lower_address(block, buddy)
        order = order + 1
    insert(block, free_list[order])
```

With constant-time free-list operations, allocation and free perform at most one operation per order, so their structural work is $O(MAX\_ORDER)$ and the configured maximum is small. The trade-off is **internal fragmentation**: a request for five pages consumes an eight-page block. External fragmentation still appears as free memory split across low orders, and compaction may be required for huge pages or other high-order requests.

#### **Slab and Object Caches**

Allocating a full page block for every small kernel object would waste memory and repeatedly initialize the same structures. A **slab-style allocator** layers typed or size-class caches over the page allocator. Each cache obtains one or more pages, divides them into fixed-size object slots, and reuses freed objects.

![A slab cache obtains page blocks, divides them into objects, and tracks occupancy.](assets/slab-object-cache.svg){fig-alt="The page allocator supplies blocks to a type-specific kernel object cache with a per-CPU fast path; full, partial, and empty slabs contain fixed-size occupied and free object slots." width="98%"}

*Figure: original explanatory diagram based on current Linux [slab allocation documentation](https://docs.kernel.org/mm/slab.html), [`/proc/kpageflags` SLAB description](https://docs.kernel.org/admin-guide/mm/pagemap.html), and the kernel.org [slab overview](https://www.kernel.org/doc/gorman/html/understand/understand011.html).*

The layers solve different granularity problems:

| Layer | Allocation unit | Optimizes | Typical cost |
|---|---|---|---|
| Buddy/page allocator | powers-of-two blocks of pages | physical contiguity and fast coalescing | power-of-two waste and high-order fragmentation |
| Slab/object cache | fixed-size kernel objects within slabs | reuse, initialization, alignment, small-object locality | metadata, partial slabs, cache-specific fragmentation |
| User allocator | application blocks within process mappings | language/runtime allocation patterns | process-local fragmentation and synchronization |

Linux commonly uses the term **SLUB** for its default slab implementation, while “slab allocator” remains the architectural family. Per-CPU freelists make common allocation paths fast and reduce shared-lock traffic; partial-slab lists allow reuse across CPUs or nodes. Debugging and hardening can add red zones, poisoning, randomized freelists, or metadata checks, trading speed and memory for error detection and exploit resistance.

Object caches are not automatically leak-proof. A reference held forever keeps an object live regardless of allocator quality, while a workload with awkward size classes can leave partially occupied slabs. `/proc/slabinfo`, `slabtop`, and kernel memory accounting help distinguish user RSS growth from kernel-object growth.

### **Huge Pages, NUMA, and Modern Memory Systems**

Base pages simplify fine-grained allocation and protection, but a large working set can exceed TLB reach and require many page-table entries. **Huge pages** map a larger contiguous range with one translation. The previous chapter covered their translation benefit; here the important question is how they are materialized and placed.

Linux exposes two broad mechanisms:

- **HugeTLB** pages are explicitly reserved and managed through a dedicated interface;
- **Transparent Huge Pages (THP)** allow the kernel to allocate, promote, split, and in supported configurations demote page sizes without requiring every application to manage a reserved pool.

THP can allocate a large page at fault time or let a background mechanism such as `khugepaged` collapse suitable base pages. Current kernels can support multiple THP sizes depending on architecture and configuration. Promotion is best effort: alignment, sharing, swapped pages, fragmentation, policy, and cgroup limits can prevent it. The [Linux THP documentation](https://docs.kernel.org/admin-guide/mm/transhuge.html) explicitly warns that a large page can waste memory when an application touches only a tiny part of a large region.

On a **Non-Uniform Memory Access (NUMA)** system, every CPU can access all ordinary memory, but latency and bandwidth depend on the memory node. Linux normally attempts local allocation and then follows a zonelist fallback order. The CPU that performs first touch often influences where an anonymous page is initially allocated.

![Huge-page promotion and NUMA placement solve different parts of memory cost.](assets/hugepage-numa-placement.svg){fig-alt="Base pages may collapse into a transparent huge page to reduce translation entries, while two NUMA nodes show fast local memory, a slower remote interconnect, first-touch placement, and the possibility that tasks migrate away from their pages." width="98%"}

*Figure: original explanatory diagram based on Linux [Transparent Hugepage Support](https://docs.kernel.org/admin-guide/mm/transhuge.html), [NUMA overview](https://docs.kernel.org/mm/numa.html), and [NUMA memory policy](https://docs.kernel.org/admin-guide/mm/numa_memory_policy.html). Exact page sizes and available policies are architecture- and kernel-dependent.*

Huge pages and NUMA interact. A huge allocation needs a larger suitable contiguous block on one node; compaction or fallback can increase fault latency or place memory remotely. A task can later migrate while its pages remain on the original node. Automatic NUMA balancing, explicit policy through `numactl`/`mbind()`, CPU affinity, and page migration can improve locality, but migration itself copies data and changes mappings.

| Choice | Potential gain | Potential harm | Measure |
|---|---|---|---|
| Base pages | fine-grained faults, reclaim, and COW | lower TLB reach and more PTEs | TLB misses, page-table size, fault count |
| THP | fewer translations, larger sequential reach | larger zero/copy/split cost, wasted memory, compaction stalls | `AnonHugePages`, THP fault/collapse counters, latency |
| Local NUMA placement | lower latency and more local bandwidth | capacity imbalance if one node fills first | `numastat`, `/proc/PID/numa_maps`, hardware counters |
| Interleaving | spreads bandwidth and capacity | some accesses are inevitably remote | bandwidth distribution and application throughput |
| Migration | repairs task/page mismatch | copy cost and mapping disruption | migration counters and before/after locality |

The right policy follows access shape. A dense, long-lived analytical array may benefit from huge local pages. A sparse region with unpredictable first-touch and frequent COW can lose more from coarse pages than it gains from TLB reach.

### **Observing Memory Behavior in Linux**

Memory diagnosis should begin with a symptom and combine four views:

1. **process footprint**: which mappings are resident, private, shared, anonymous, file-backed, huge, or swapped;
2. **system composition**: how RAM is divided among anonymous pages, page cache, slab, page tables, and reclaimable state;
3. **activity and pressure**: whether faults, reclaim, writeback, swap, or stalls are increasing over time;
4. **placement**: whether pages are local to the CPUs using them and whether page size matches expectations.

![Linux memory diagnosis triangulates process, system, pressure, and placement evidence.](assets/linux-memory-observability.svg){fig-alt="An observability workflow starts from a latency or memory symptom, gathers process smaps, system meminfo and vmstat, pressure stall information, NUMA and perf views, then correlates a time series before testing a hypothesis." width="98%"}

*Figure: original explanatory workflow based on Linux [`/proc` documentation](https://www.kernel.org/doc/html/latest/filesystems/proc.html), [PSI documentation](https://docs.kernel.org/accounting/psi.html), and the [`proc_pid_smaps(5)` manual](https://man7.org/linux/man-pages/man5/proc_pid_smaps.5.html).*

The following read-only commands form a useful first pass on Linux:

```bash
pid=${1:-$$}

# Process totals. PSS apportions shared pages; RSS counts each mapping fully.
grep -E '^(Rss|Pss|Private_|Shared_|Anonymous|AnonHugePages|Swap):' \
  "/proc/$pid/smaps_rollup"

# System composition and commitment.
grep -E '^(MemTotal|MemAvailable|Cached|AnonPages|Slab|PageTables|SwapTotal|SwapFree|CommitLimit|Committed_AS):' \
  /proc/meminfo

# Pressure reports time lost to contention, not just capacity consumption.
cat /proc/pressure/memory

# Time series: run queue, free memory, paging, block I/O, and CPU activity.
vmstat 1 5

# Optional on NUMA systems when numactl tools are installed.
command -v numastat >/dev/null && numastat -p "$pid"
```

Interpretation needs care:

- `VmSize` or mapping `Size` is not resident memory;
- RSS double-counts a shared page in each mapping, while PSS apportions it;
- `MemFree` excludes useful reclaimable cache, so `MemAvailable` is usually the better capacity estimate;
- minor faults can be normal first-touch/COW behavior;
- major faults indicate storage was required for the requested page, but one snapshot does not show rate;
- `si`/`so` in `vmstat` are activity rates, while total swap occupancy is a state;
- memory PSI `some` means at least some tasks are stalled, while `full` indicates all non-idle work is simultaneously stalled and is stronger evidence of lost progress;
- `/proc/PID/smaps` and related files are snapshots and can race with mapping changes.

A disciplined investigation records a time series around the latency or OOM event, then tests one causal hypothesis. For example:

```text
Observation: RSS and active anonymous memory rise.
Observation: pgscan/pgsteal and swap-in/out rise soon afterward.
Observation: memory full PSI and request latency rise together.
Hypothesis: the workload's active anonymous set exceeds its memory budget.
Test: reduce concurrency or raise the controlled cgroup budget, then repeat.
```

This is stronger than concluding “Linux used all RAM.” A page cache near capacity may be healthy; a small RSS with severe remote NUMA traffic may be slow; a high commitment may be harmless until first touch; and a one-time swap value may describe old cold pages rather than current churn.

### **Comparison and Summary**

The chapter's mechanisms act at different moments in a page's lifecycle:

| Mechanism | Trigger or input | Main decision | Result | Principal trade-off |
|---|---|---|---|---|
| Demand paging | first access to nonresident legal page | materialize now or fail | PTE points to supplied contents | lower eager work versus first-touch latency |
| Zero-fill-on-demand | first read/write of anonymous page | share zero page or allocate cleared frame | secure zero semantics | saved frames versus write-fault cost |
| Copy-on-write | write to deliberately shared private page | copy, or upgrade if exclusive | writer receives private writable state | avoids unused copies versus burst copy cost |
| Page cache | file object and offset | reuse, read, dirty, or write back cached contents | coherent file-backed resident page | shared reuse versus reclaim/writeback complexity |
| Replacement | frame shortage | choose eligible resident victim | frame becomes reusable | preserving locality versus tracking overhead |
| Working-set/PFF control | recent references or fault rate | adjust residency/concurrency | less destructive churn | estimation window and responsiveness |
| Swap | cold anonymous page under pressure | retain data outside RAM | frame freed, contents recoverable | capacity relief versus high access latency |
| Overcommit | reservation/commit request | accept potential future use | promise recorded or request rejected | utilization versus late failure risk |
| Buddy allocator | page-block request | split/merge powers-of-two blocks | contiguous page block | speed versus fragmentation |
| Slab cache | small kernel-object request | reuse a typed/size-class slot | initialized object | fast locality versus cache metadata/partial slabs |
| THP | dense aligned region and policy | allocate/promote larger page | greater TLB reach | translation savings versus coarse fault/reclaim cost |
| NUMA policy | allocation and task placement | choose memory node or migrate | local or remote physical placement | locality versus balance and migration cost |

One legal memory access can now be traced end to end:

1. The CPU finds no usable translation and raises a precise fault.
2. The kernel revalidates the region and requested permission.
3. Region metadata identifies anonymous, COW, file, or swap backing.
4. A resident shared page is reused, or a frame is requested.
5. Under pressure, reclaim filters candidates and applies replacement policy.
6. Dirty file data may need writeback; anonymous contents may need swap.
7. The page allocator supplies a suitable block, possibly after reclaim or compaction.
8. The handler clears, copies, or loads the page and resolves concurrent fault races.
9. It installs the PTE, refreshes stale translation state, and restarts the instruction.
10. Later aging, pressure, page size, and NUMA policy determine how long and where the page remains resident.

Several misconceptions can now be rejected precisely:

- a page fault is not synonymous with disk I/O;
- a minor fault is not necessarily negligible;
- a successful large `malloc()` does not prove all promised pages are resident or guaranteed under every overcommit policy;
- full RAM is not itself thrashing, because cache is deliberately reclaimable;
- swap occupancy is not the same as active swap churn;
- FIFO residence age is not LRU access recency;
- exact LRU is not the literal policy of a scalable modern kernel;
- reclaiming many base pages does not guarantee a contiguous high-order block;
- huge pages and local NUMA placement are workload-dependent optimizations, not universal wins;
- RSS alone cannot attribute shared memory or explain stall time.

Address translation gave the process a protected and sparse namespace. Demand paging turns that namespace into a dynamic working set; replacement and allocation keep the physical machine usable; observability tells us whether those policies preserve progress. The next chapter follows the blocking side of this story into device controllers, interrupts, DMA, drivers, and event-driven I/O.
